# 01 — Exploring CFBD play-by-play

First look at the raw data everything else in this project is built from: one week of play-by-play from the CollegeFootballData API. Goal is just to understand the schema before we derive any metrics.

In [1]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent / "src"))

import pandas as pd
from cfb_analytics.client import get_plays, get_teams

pd.set_option("display.max_columns", 50)

In [2]:
plays = get_plays(2024, week=1)
print(plays.shape)
plays.head()

(22356, 28)


,gameId,driveId,id,driveNumber,playNumber,offense,offenseConference,offenseScore,defense,defenseConference,defenseScore,home,away,period,offenseTimeouts,defenseTimeouts,yardline,yardsToGoal,down,distance,yardsGained,scoring,playType,playText,ppa,wallclock,clock.minutes,clock.seconds
0,401628323,4016283231,401628323101854401,1,2,Clemson,ACC,0,Georgia,SEC,0,Georgia,Clemson,1,3.0,3.0,75,75,1,10,0,False,Pass Incompletion,Cade Klubnik pass incomplete to Phil Mafah,-0.702970,2024-08-31T16:09:01.000Z,14,55
1,401628323,4016283231,401628323101855001,1,3,Clemson,ACC,0,Georgia,SEC,0,Georgia,Clemson,1,3.0,3.0,75,75,2,10,-5,False,Penalty,"Clemson Penalty, Illegal Motion (-5 Yards) to ...",NaN,2024-08-31T16:10:09.000Z,14,49
2,401628323,4016283231,401628323101849902,1,1,Georgia,SEC,0,Clemson,ACC,0,Georgia,Clemson,1,3.0,3.0,35,65,1,10,0,False,Kickoff,Peyton Woodring kickoff for 65 yds for a touch...,NaN,2024-08-31T16:06:09.000Z,15,0
3,401628323,4016283231,401628323101857501,1,4,Clemson,ACC,0,Georgia,SEC,0,Georgia,Clemson,1,3.0,3.0,80,80,2,15,0,False,Pass Incompletion,Cade Klubnik pass incomplete to Jake Briningstool,-0.149039,2024-08-31T16:10:38.000Z,14,24
4,401628323,4016283231,401628323101859201,1,5,Clemson,ACC,0,Georgia,SEC,0,Georgia,Clemson,1,3.0,3.0,80,80,3,15,5,False,Penalty,"Georgia Penalty, Defensive Offside (Mykel Will...",NaN,2024-08-31T16:11:49.000Z,14,7


Every row is one play. Note `down`/`distance` are only populated on real scrimmage plays — kickoffs, timeouts, and end-of-period markers all show up as rows too, with `down` as NaN.

In [3]:
plays["playType"].value_counts()

playType
Rush                             8132
Pass Reception                   4182
Pass Incompletion                2807
Penalty                          1153
Punt                             1145
Timeout                           971
Kickoff                           938
Kickoff Return (Offense)          444
Sack                              436
Rushing Touchdown                 416
Passing Touchdown                 393
Field Goal Good                   267
End Period                        263
Fumble Recovery (Own)             136
Pass Interception Return          132
End of Half                       119
End of Game                       117
Field Goal Missed                  95
Fumble Recovery (Opponent)         86
Interception                       30
Uncategorized                      20
Interception Return Touchdown      17
Fumble Return Touchdown            14
Blocked Punt                       13
Safety                              9
Blocked Field Goal                  6
Kic

In [4]:
# ppa: CFBD's own predicted-points-added value per play (their EPA equivalent).
# We'll lean on this later for defensive efficiency instead of re-deriving EPA from scratch.
plays[plays["down"].notna()][["offense", "defense", "down", "distance", "yardsGained", "ppa", "playType"]].describe(include="all")

,offense,defense,down,distance,yardsGained,ppa,playType
count,22356,22356,22356.000000,22356.000000,22356.000000,16777.000000,22356
unique,252,252,NaN,NaN,NaN,NaN,31
top,SMU,North Alabama,NaN,NaN,NaN,NaN,Rush
freq,201,193,NaN,NaN,NaN,NaN,8132
mean,NaN,NaN,1.881285,8.294507,5.797683,0.109563,NaN
std,NaN,NaN,1.098522,4.292269,11.228458,1.460022,NaN
min,NaN,NaN,0.000000,-11.000000,-91.000000,-11.486173,NaN
25%,NaN,NaN,1.000000,5.000000,0.000000,-0.598306,NaN
50%,NaN,NaN,2.000000,10.000000,2.000000,-0.118270,NaN
75%,NaN,NaN,3.000000,10.000000,8.000000,0.795914,NaN


In [5]:
teams = get_teams(2024)
one_team = plays[plays["offense"] == "Georgia"]
one_team[["down", "distance", "yardsGained", "playType"]].head(20)

,down,distance,yardsGained,playType
2,1,10,0,Kickoff
7,1,10,11,Pass Reception
8,1,10,0,Rush
9,2,10,0,Pass Reception
10,3,10,13,Rush
11,1,10,2,Pass Reception
12,2,8,0,Rush
13,3,8,0,Pass Incompletion
14,4,8,0,Punt
22,1,10,1,Rush


Next: [02_build_success_rate.ipynb](02_build_success_rate.ipynb) derives the success-rate metric from these fields, by hand first, then checks it against `cfb_analytics.metrics`.